In [3]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from scipy.stats import zscore
from scipy.fft import fft
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

DATASET_PATH = "HardwareTrojanVulnerabilityAnalysisDataset"

IMG_SIZE = 128

OUTLIER_THRESHOLD = 3

TEST_SIZE = 0.2

def moving_average(signal, window_size=5):
    return np.convolve(signal, np.ones(window_size)/window_size, mode='valid')

def preprocess_image(image_path):

    img = cv2.imread(image_path)

    if img is None:
        return None

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    equalized = cv2.equalizeHist(gray)

    h, w = equalized.shape
    cropped = equalized[int(h*0.1):int(h*0.9), int(w*0.1):int(w*0.9)]

    signal = cropped.flatten()

    filtered_signal = moving_average(signal)

    return filtered_signal

features = []
labels = []

print("Loading dataset...")

for label in os.listdir(DATASET_PATH):

    folder = os.path.join(DATASET_PATH, label)

    if not os.path.isdir(folder):
        continue

    print("Processing class:", label)

    for img_name in tqdm(os.listdir(folder)):

        img_path = os.path.join(folder, img_name)

        processed = preprocess_image(img_path)

        if processed is not None:
            features.append(processed)
            labels.append(label)

features = np.array(features)
labels = np.array(labels)

print("Total samples:", len(features))

z_scores = np.abs(zscore(features))
mask = (z_scores < OUTLIER_THRESHOLD).all(axis=1)

features = features[mask]
labels = labels[mask]

print("Samples after outlier removal:", len(features))

spectral_features = []

print("Extracting spectral features...")

for signal in tqdm(features):

    fft_vals = np.abs(fft(signal))

    mean_freq = np.mean(fft_vals)

    spectral_centroid = np.sum(np.arange(len(fft_vals)) * fft_vals) / np.sum(fft_vals)

    bandwidth = np.sqrt(
        np.sum(((np.arange(len(fft_vals)) - spectral_centroid)**2) * fft_vals)
        / np.sum(fft_vals)
    )

    flatness = np.exp(np.mean(np.log(fft_vals + 1e-10))) / np.mean(fft_vals)

    spectral_features.append([
        mean_freq,
        spectral_centroid,
        bandwidth,
        flatness
    ])

spectral_features = np.array(spectral_features)

print("Feature shape:", spectral_features.shape)

scaler = StandardScaler()
X = scaler.fit_transform(spectral_features)

encoder = LabelEncoder()
y = encoder.fit_transform(labels)

print("Classes:", encoder.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=y
)

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))

np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)
np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)

print("Preprocessing completed")

Loading dataset...
Processing class: HighVulnerableClass


100%|██████████| 3152/3152 [00:00<00:00, 4298.53it/s]


Processing class: MediumVulnerableClass


100%|██████████| 5941/5941 [00:01<00:00, 3984.10it/s]


Processing class: LowVulnerableClass


100%|██████████| 3317/3317 [00:00<00:00, 4213.57it/s]


Total samples: 12410
Samples after outlier removal: 12385
Extracting spectral features...


100%|██████████| 12385/12385 [00:03<00:00, 4004.92it/s]

Feature shape: (12385, 4)
Classes: ['HighVulnerableClass' 'LowVulnerableClass' 'MediumVulnerableClass']
Train samples: 9908
Test samples: 2477
Preprocessing completed


In [ ]:
# approach one is training a deep neural network on the features extracted from the image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

X_train = np.load("X_train.npy")
X_test = np.load("X_test.npy")
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")

print("Data shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")


model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(4,)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dense(3, activation='softmax')  
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel Summary:")
model.summary()

In [4]:
# approach two is training an SVC
# approach three is training a RandomForestClassfier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train = np.load("X_train.npy")
X_test = np.load("X_test.npy")
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")

print("Shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_pred)

print("\n=== SVM Results ===")
print(f"Accuracy: {svm_acc:.4f}")
print("Classification Report:")
print(classification_report(y_test, svm_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, svm_pred))

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print("\n=== Random Forest Results ===")
print(f"Accuracy: {rf_acc:.4f}")
print("Classification Report:")
print(classification_report(y_test, rf_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\n=== Quick Comparison ===")
print(f"SVM Accuracy:          {svm_acc:.4f}")
print(f"Random Forest Accuracy:{rf_acc:.4f}")
print("Best model:", "SVM" if svm_acc >= rf_acc else "Random Forest")

Shapes:
X_train: (9908, 4), y_train: (9908,)
X_test: (2477, 4), y_test: (2477,)

=== SVM Results ===
Accuracy: 0.5083
Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.05      0.09       628
           1       0.56      0.18      0.28       663
           2       0.50      0.93      0.65      1186

    accuracy                           0.51      2477
   macro avg       0.62      0.39      0.34      2477
weighted avg       0.59      0.51      0.41      2477

Confusion Matrix:
[[  30   25  573]
 [   1  121  541]
 [   7   71 1108]]

=== Random Forest Results ===
Accuracy: 0.5018
Classification Report:
              precision    recall  f1-score   support

           0       0.43      0.26      0.32       628
           1       0.48      0.43      0.45       663
           2       0.53      0.67      0.59      1186

    accuracy                           0.50      2477
   macro avg       0.48      0.45      0.46      2477
weighted

In [ ]:
# Conclusion is that a standard ml model like svc or random forest classifier doesnt work on these features 
# will try a transfer learning approach 